<h2><u>Mumbai Metrics</u></h2>
<img src="mumbai.jpg" width="250" height="150" align="right">
<h4> The above notebook contains a <b>Mumbai City House Price Prediction Analysis</b> which contains <b>76,038 records</b>
and a total of <b>13 columns</b>, also the completeness of the data is <b>100%</b>. In short, it is a comprehensive real estate dataset containing basic property information (location, size, price, age) and synthetic market analytics (ROI, demand, volatility, liquidity scores).</h4>
<br>
<b><h4><a href="https://www.kaggle.com/datasets/tilakjain/mumbai-house-data/data?select=mumbai_house_data.csv">Link to the Dataset</a></h4></b>
    

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Reading the .csv file
df = pd.read_csv('mumbai_house_data.csv')

#Undersatnding the dataset and its datatypes
print("\nDataset Shape:", df.shape)
print("\nData Info & Datatypes:", df.info())
print("\nDescription:", df.describe())
print("\nFirst 5 Rows:", df.head())
print("\nLast 5 Rows:", df.tail())

#Checking for missing values
print("\nMissing values count per column:", df.isnull().sum())
print("\nNo missing values")


Dataset Shape: (76038, 13)
<class 'pandas.DataFrame'>
RangeIndex: 76038 entries, 0 to 76037
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   bhk                       76038 non-null  int64  
 1   type                      76038 non-null  str    
 2   locality                  76038 non-null  str    
 3   area                      76038 non-null  int64  
 4   price                     76038 non-null  float64
 5   price_unit                76038 non-null  str    
 6   region                    76038 non-null  str    
 7   status                    76038 non-null  str    
 8   age                       76038 non-null  str    
 9   expected_roi(%)           76038 non-null  float64
 10  demand_indicator          76038 non-null  float64
 11  market_volatitlity_score  76038 non-null  float64
 12  property_liquidity_index  76038 non-null  float64
dtypes: float64(5), int64(2), str(6)
memory usage

In [18]:
#Checking for Duplicate values
print("Dupliicate Values:", df.duplicated())

print("\nNo duplicate values")

Dupliicate Values: 0        False
1        False
2        False
3        False
4        False
         ...  
76033    False
76034    False
76035    False
76036    False
76037    False
Length: 76038, dtype: bool

No duplicate values


In [34]:
initial_count = len(df)
print("Initial Count:", initial_count)

Initial Count: 76038


In [35]:
#Handling text anomalies
if df['price'].dtype == 'O':
    df['price'] = df['price'].str.replace(r'[^\d.]', '', regex=True)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

if df['area'].dtype == 'O':
    df['area'] = df['area'].str.replace(r'[^\d.]', '', regex=True)
    df['area'] = pd.to_numeric(df['area'], errors='coerce')

# 3. Handle Missing Values, replacing location columns with 'Unknow' value or dropping the,
df['locality'] = df['locality'].fillna('Unknown')

#Filter Outliers/Data Entry Errors
Q1 = df['price'].quantile(0.25)
print("First Quartile Value:", Q1)
Q3 = df['price'].quantile(0.75)
print("Third Quartile Value", Q3)
IQR = Q3 - Q1
print("Interquartile Range:", IQR)

# Keeping rows within acceptable economic variance
lower_bound = max(0, Q1 - 3 * IQR)
print("Lower Bound value:",lower_bound)
upper_bound = Q3 + 3 * IQR
print("Upper Bound value:",upper_bound)

df.drop_duplicates(inplace=True)
print(f"Removed {initial_count - len(df)} duplicate rows.")
df_cleaned = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)].copy()
print(f"Cleaned dataset size after outlier removal: {df_cleaned.shape[0]} rows.")

First Quartile Value: 1.75
Third Quartile Value 59.0
Interquartile Range: 57.25
Lower Bound value: 0
Upper Bound value: 230.75
Removed 0 duplicate rows.
Cleaned dataset size after outlier removal: 76038 rows.


In [1]:
# Create a copy for cleaning
df_cleaned = df.copy()

# 1. Clean Numeric Fields (Strip spaces/symbols if text-encoded)
for col in ['price', 'area', 'expected_roi(%)', 'market_volatility_score', 'property_liquidity_index']:
    if df_cleaned[col].dtype == 'O':
        df_cleaned[col] = df_cleaned[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# 2. Handle Missing Values using median strategies
num_cols = ['price', 'area', 'bhk', 'age', 'expected_roi(%)', 'market_volatility_score', 'property_liquidity_index']
for col in num_cols:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# For categorical columns, fill missing with 'Unknown'
df_cleaned['region'] = df_cleaned['region'].fillna('Unknown')
df_cleaned['status'] = df_cleaned['status'].fillna('Unknown')
df_cleaned['demand_indicator'] = df_cleaned['demand_indicator'].fillna('Medium')

# 3. Encode Categorical Data to Numerical for Correlation Mapping
# Label encode region and status
le_region = LabelEncoder()
df_cleaned['region_encoded'] = le_region.fit_transform(df_cleaned['region'].astype(str))

le_status = LabelEncoder()
df_cleaned['status_encoded'] = le_status.fit_transform(df_cleaned['status'].astype(str))

# Ordinal map 'demand_indicator' if it contains tiers (Low, Medium, High)
demand_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
# If it's already numeric, this step safely keeps it as is or maps text tiers
if df_cleaned['demand_indicator'].dtype == 'O':
    df_cleaned['demand_indicator_encoded'] = df_cleaned['demand_indicator'].map(demand_mapping).fillna(2)
else:
    df_cleaned['demand_indicator_encoded'] = df_cleaned['demand_indicator']

print("Data cleaning and numerical mapping complete.")

NameError: name 'df' is not defined